# 02 — SQL Data Cleaning & Master Table (Pure SQL)
### Olist E-Commerce — Delivery & Review Risk Analysis

Notebook 1 did the Python/pandas pass. **This notebook rebuilds the cleaning and joining logic entirely in SQL** — across all 9 raw CSVs — so the SQL skills are genuinely demonstrated, not just the final analysis queries.

What this notebook does, all in SQL against `olist.db`:
1. Loads the **geolocation** table into the database (the one table Notebook 1 didn't need) — now all 9 source files live in SQL
2. Row-counts and null-profiles every table with SQL
3. Checks referential integrity with `LEFT JOIN ... IS NULL` across every relationship, including geolocation
4. Deduplicates reviews per order with a **window function** (`ROW_NUMBER() OVER (PARTITION BY ...)`)
5. Aggregates payments per order and geolocation per zip code with **`GROUP BY`**
6. Assembles one clean, order-level **master table** using **CTEs + multiple joins**
7. Validates the SQL-built table against the pandas-built one from Notebook 1


In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DB_PATH = '../Data/processed/olist.db'
conn = sqlite3.connect(DB_PATH)
print('Connected to', DB_PATH)


Connected to ../Data/processed/olist.db


## 1. Complete the database — load `geolocation`
This is the only one of the 9 CSVs Notebook 1 didn't load into SQLite. It's large (~1M rows, many rows per zip code) so we load it raw and let SQL do the aggregation later.

In [2]:
geoloc = pd.read_csv('../Data/olist_geolocation_dataset.csv')
geoloc.to_sql('geolocation', conn, if_exists='replace', index=False)
conn.execute('CREATE INDEX IF NOT EXISTS idx_geo_zip ON geolocation(geolocation_zip_code_prefix)')
conn.commit()
print(f'Loaded geolocation: {len(geoloc):,} rows')


Loaded geolocation: 1,000,163 rows


## 2. Table inventory — row counts, pure SQL
One query, all 8 tables, using `UNION ALL` rather than 8 separate queries.

In [3]:
pd.read_sql("""
    SELECT 'orders' AS table_name, COUNT(*) AS n_rows FROM orders
    UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
    UNION ALL SELECT 'order_payments', COUNT(*) FROM order_payments
    UNION ALL SELECT 'order_reviews', COUNT(*) FROM order_reviews
    UNION ALL SELECT 'products', COUNT(*) FROM products
    UNION ALL SELECT 'customers', COUNT(*) FROM customers
    UNION ALL SELECT 'sellers', COUNT(*) FROM sellers
    UNION ALL SELECT 'geolocation', COUNT(*) FROM geolocation
""", conn)


,table_name,n_rows
0,orders,99441
1,order_items,112650
2,order_payments,103886
3,order_reviews,99224
4,products,32951
5,customers,99441
6,sellers,3095
7,geolocation,1000163


## 3. SQL data-quality checks — nulls in key columns
Same idea as Notebook 1's pandas `.isnull().sum()`, done here with SQL's `COUNT(col) vs COUNT(*)` pattern.

In [4]:
pd.read_sql("""
    SELECT
        COUNT(*) AS total_orders,
        COUNT(*) - COUNT(order_approved_at) AS missing_approved_at,
        COUNT(*) - COUNT(order_delivered_carrier_date) AS missing_carrier_date,
        COUNT(*) - COUNT(order_delivered_customer_date) AS missing_delivered_date
    FROM orders
""", conn)


,total_orders,missing_approved_at,missing_carrier_date,missing_delivered_date
0,99441,160,1783,2965


In [5]:
pd.read_sql("""
    SELECT
        COUNT(*) AS total_products,
        COUNT(*) - COUNT(product_category_name) AS missing_category,
        COUNT(*) - COUNT(product_weight_g) AS missing_weight
    FROM products
""", conn)


,total_products,missing_category,missing_weight
0,32951,0,2


## 4. Referential integrity — orphan checks via `LEFT JOIN ... IS NULL`
This is the SQL-native way to answer "does every foreign key resolve?" — join the child to the parent and see what comes back `NULL`.

In [6]:
pd.read_sql("""
    SELECT 'items -> orders' AS relationship, COUNT(*) AS orphans
    FROM order_items i LEFT JOIN orders o ON i.order_id = o.order_id
    WHERE o.order_id IS NULL

    UNION ALL
    SELECT 'items -> products', COUNT(*)
    FROM order_items i LEFT JOIN products p ON i.product_id = p.product_id
    WHERE p.product_id IS NULL

    UNION ALL
    SELECT 'items -> sellers', COUNT(*)
    FROM order_items i LEFT JOIN sellers s ON i.seller_id = s.seller_id
    WHERE s.seller_id IS NULL

    UNION ALL
    SELECT 'payments -> orders', COUNT(*)
    FROM order_payments pay LEFT JOIN orders o ON pay.order_id = o.order_id
    WHERE o.order_id IS NULL

    UNION ALL
    SELECT 'reviews -> orders', COUNT(*)
    FROM order_reviews r LEFT JOIN orders o ON r.order_id = o.order_id
    WHERE o.order_id IS NULL

    UNION ALL
    SELECT 'orders -> customers', COUNT(*)
    FROM orders o LEFT JOIN customers c ON o.customer_id = c.customer_id
    WHERE c.customer_id IS NULL

    UNION ALL
    SELECT 'customers -> geolocation (zip)', COUNT(*)
    FROM customers c
    LEFT JOIN (SELECT DISTINCT geolocation_zip_code_prefix FROM geolocation) g
        ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
    WHERE g.geolocation_zip_code_prefix IS NULL

    UNION ALL
    SELECT 'sellers -> geolocation (zip)', COUNT(*)
    FROM sellers s
    LEFT JOIN (SELECT DISTINCT geolocation_zip_code_prefix FROM geolocation) g
        ON s.seller_zip_code_prefix = g.geolocation_zip_code_prefix
    WHERE g.geolocation_zip_code_prefix IS NULL
""", conn)


,relationship,orphans
0,items -> orders,0
1,items -> products,0
2,items -> sellers,0
3,payments -> orders,0
4,reviews -> orders,0
5,orders -> customers,0
6,customers -> geolocation (zip),278
7,sellers -> geolocation (zip),7


**Reading this result:** the order/item/payment/review/customer relationships should all resolve at 0 orphans — confirmed already in Notebook 1. `customer/seller -> geolocation` orphans are expected to be non-zero: not every zip code prefix in Brazil's postal system has a geolocation sample in this dataset. We'll handle that with a `LEFT JOIN` (keep the order, allow a null lat/lng) rather than dropping orders over it.

## 5. Deduplicate reviews per order — window function
An order can have more than one review row (e.g. a customer resubmits). `ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY review_answer_timestamp DESC)` keeps only the latest review per order — no `GROUP BY` collapses this as cleanly as a window function does.

In [7]:
conn.execute("DROP TABLE IF EXISTS order_reviews_dedup")
conn.execute("""
    CREATE TABLE order_reviews_dedup AS
    WITH ranked AS (
        SELECT
            order_id,
            review_score,
            review_creation_date,
            ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY review_answer_timestamp DESC
            ) AS rn
        FROM order_reviews
    )
    SELECT order_id, review_score, review_creation_date
    FROM ranked
    WHERE rn = 1
""")
conn.commit()

pd.read_sql("SELECT COUNT(*) AS deduped_review_rows FROM order_reviews_dedup", conn)


,deduped_review_rows
0,98673


## 6. Aggregate payments per order — `GROUP BY`
An order can be split across multiple payment rows (e.g. part credit card, part voucher). Roll them up to one row per order.

In [8]:
conn.execute("DROP TABLE IF EXISTS order_payments_agg")
conn.execute("""
    CREATE TABLE order_payments_agg AS
    SELECT
        order_id,
        SUM(payment_value) AS total_payment,
        COUNT(*) AS n_payment_rows,
        MAX(payment_installments) AS max_installments
    FROM order_payments
    GROUP BY order_id
""")
conn.commit()

pd.read_sql("SELECT * FROM order_payments_agg LIMIT 5", conn)


,order_id,total_payment,n_payment_rows,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3


## 7. Aggregate geolocation to zip level — `GROUP BY` + `AVG`
The raw table has many lat/lng samples per zip prefix. Average them down to one row per zip so joining to customers/sellers doesn't fan out the row count.

In [9]:
conn.execute("DROP TABLE IF EXISTS geo_zip_agg")
conn.execute("""
    CREATE TABLE geo_zip_agg AS
    SELECT
        geolocation_zip_code_prefix AS zip,
        AVG(geolocation_lat) AS avg_lat,
        AVG(geolocation_lng) AS avg_lng
    FROM geolocation
    GROUP BY geolocation_zip_code_prefix
""")
conn.commit()

pd.read_sql("SELECT COUNT(*) AS distinct_zips FROM geo_zip_agg", conn)


,distinct_zips
0,19015


## 8. Build the order-level master table — CTEs + multiple joins
This is the core SQL deliverable: a single query using several CTEs (one per aggregation problem — items, main category, payments, reviews) that are then joined together against `orders` and `customers`. Everything upstream (dedup, aggregation) was already materialized as tables in steps 5–7, so this final query focuses purely on the joins.

In [10]:
conn.execute("DROP TABLE IF EXISTS master_orders_sql")
conn.execute("""
    CREATE TABLE master_orders_sql AS
    WITH order_items_agg AS (
        SELECT
            order_id,
            SUM(price) AS item_price,
            SUM(freight_value) AS freight_value,
            COUNT(*) AS n_items
        FROM order_items
        GROUP BY order_id
    ),
    items_ranked AS (
        SELECT
            order_id,
            product_id,
            ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY price DESC) AS item_rank
        FROM order_items
    ),
    main_category AS (
        SELECT
            ir.order_id,
            p.product_category_name_english AS main_category
        FROM items_ranked ir
        JOIN products p ON ir.product_id = p.product_id
        WHERE ir.item_rank = 1
    )
    SELECT
        o.order_id,
        o.customer_id,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        o.is_delivered,
        o.delivery_time_days,
        o.delay_days,
        o.is_late,
        c.customer_city,
        c.customer_state,
        cg.avg_lat  AS customer_lat,
        cg.avg_lng  AS customer_lng,
        oia.item_price,
        oia.freight_value,
        oia.n_items,
        (oia.item_price + oia.freight_value) AS order_revenue,
        pa.total_payment,
        pa.n_payment_rows,
        rd.review_score,
        mc.main_category
    FROM orders o
    LEFT JOIN customers          c   ON o.customer_id = c.customer_id
    LEFT JOIN geo_zip_agg        cg  ON c.customer_zip_code_prefix = cg.zip
    LEFT JOIN order_items_agg    oia ON o.order_id = oia.order_id
    LEFT JOIN main_category      mc  ON o.order_id = mc.order_id
    LEFT JOIN order_payments_agg pa  ON o.order_id = pa.order_id
    LEFT JOIN order_reviews_dedup rd ON o.order_id = rd.order_id
""")
conn.commit()
print('master_orders_sql created.')


master_orders_sql created.


## 9. Validate the SQL-built master table
Row count should equal `orders` (one row per order — no fan-out from the joins), and the numbers should line up with the pandas version from Notebook 1.

In [11]:
check = pd.read_sql("""
    SELECT
        (SELECT COUNT(*) FROM orders) AS n_orders,
        (SELECT COUNT(*) FROM master_orders_sql) AS n_master_rows,
        (SELECT COUNT(*) FROM master_orders_sql WHERE order_revenue IS NOT NULL) AS rows_with_revenue,
        (SELECT COUNT(*) FROM master_orders_sql WHERE review_score IS NOT NULL) AS rows_with_review,
        (SELECT ROUND(AVG(delivery_time_days), 2) FROM master_orders_sql WHERE is_delivered = 1) AS avg_delivery_days
""", conn)
check


,n_orders,n_master_rows,rows_with_revenue,rows_with_review,avg_delivery_days
0,99441,99441,98666,98673,12.56


In [12]:
master_sql = pd.read_sql("SELECT * FROM master_orders_sql", conn)
master_sql.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,delivery_time_days,delay_days,is_late,customer_city,customer_state,customer_lat,customer_lng,item_price,freight_value,n_items,order_revenue,total_payment,n_payment_rows,review_score,main_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,1,8.436574,-7.107488,0,sao paulo,SP,-23.576983,-46.587161,29.99,8.72,1.0,38.71,38.71,3.0,4.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13 00:00:00,1,13.782037,-5.355729,0,barreiras,BA,-12.177924,-44.660711,118.70,22.76,1.0,141.46,141.46,1.0,4.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04 00:00:00,1,9.394213,-17.245498,0,vianopolis,GO,-16.745150,-48.514783,159.90,19.22,1.0,179.12,179.12,1.0,5.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15 00:00:00,1,13.208750,-12.980069,0,sao goncalo do amarante,RN,-5.774190,-35.271143,45.00,27.20,1.0,72.20,72.20,1.0,5.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26 00:00:00,1,2.873877,-9.238171,0,santo andre,SP,-23.676370,-46.514627,19.90,8.72,1.0,28.62,28.62,1.0,5.0,stationery


**Cross-check against Notebook 1:** the row count (`n_orders` = `n_master_rows`, no join fan-out), the share of orders with revenue/review data, and the average delivery time here should match Notebook 1's pandas-built master table almost exactly — small differences (a few rows) can come from the `LEFT JOIN` on zip-level geolocation, which pandas' merge in NB1 didn't include at all. That agreement is the real proof that the SQL joins are correct, not just that they ran without erroring.

## 10. Save for downstream notebooks
Persist the SQL-built master table to CSV, and keep it in `olist.db` (already done via `CREATE TABLE`) so Notebook 3 can query it directly without re-running these CTEs.

In [13]:
master_sql.to_csv('../Data/processed/master_orders_sql.csv', index=False)
print('Saved master_orders_sql.csv —', master_sql.shape)
conn.close()


Saved master_orders_sql.csv — (99441, 22)


---
**Next:** `03_sql_business_analysis.ipynb` — the investigative SQL queries that actually answer the business question: top sellers by late-delivery rate, revenue at risk from low-rated orders, delivery time vs. review score by region, and month-over-month trends.
